Importing and checking the verison of PySpark

In [33]:
import pyspark
pyspark.__version__

'4.1.1'

1. Start a Spark Session

In [34]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StudentAnalysis") \
    .getOrCreate()

2. Load the CSV Dataset

In [35]:
df = spark.read.csv(
    "sources/synthetic_student_learning_dataset_10000.csv",
    header=True,
    inferSchema=True
)

3. View the Schema

In [36]:
df.printSchema()

root
 |-- respondent_id: integer (nullable = true)
 |-- education_level: string (nullable = true)
 |-- study_hours_per_day: integer (nullable = true)
 |-- preferred_learning_method: string (nullable = true)
 |-- main_learning_challenge: string (nullable = true)
 |-- motivation_level: string (nullable = true)
 |-- online_learning_opinion: string (nullable = true)
 |-- device_used_for_study: string (nullable = true)



4. Preview the Data

In [37]:
df.show(5, truncate=False)

+-------------+---------------+-------------------+-------------------------+---------------------------------+----------------+------------------------------------------------------+---------------------+
|respondent_id|education_level|study_hours_per_day|preferred_learning_method|main_learning_challenge          |motivation_level|online_learning_opinion                               |device_used_for_study|
+-------------+---------------+-------------------+-------------------------+---------------------------------+----------------+------------------------------------------------------+---------------------+
|1            |Undergraduate  |6                  |Practice exercises       |Time management difficulty       |Low             |Online learning is flexible and convenient            |Tablet               |
|2            |Postgraduate   |1                  |Practice exercises       |Academic workload pressure       |High            |Online learning is effective for theory-based su

5. Dataset Size Check

In [38]:
print("Number of rows:", df.count())
print("Columns:", len(df.columns))

Number of rows: 10000
Columns: 8


6. Check for Null Values per Column

In [39]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

+-------------+---------------+-------------------+-------------------------+-----------------------+----------------+-----------------------+---------------------+
|respondent_id|education_level|study_hours_per_day|preferred_learning_method|main_learning_challenge|motivation_level|online_learning_opinion|device_used_for_study|
+-------------+---------------+-------------------+-------------------------+-----------------------+----------------+-----------------------+---------------------+
|            0|              0|                  0|                        0|                      0|               0|                      0|                    0|
+-------------+---------------+-------------------+-------------------------+-----------------------+----------------+-----------------------+---------------------+



7. Check for Duplicate Rows (Entire Row)

In [40]:
duplicate_rows = df.count() - df.dropDuplicates().count()

print("Number of duplicate rows:", duplicate_rows)

Number of duplicate rows: 0


8. Get Distinct Categories for Each Categorical Column

In [41]:
categorical_columns = [
    "education_level",
    "preferred_learning_method",
    "main_learning_challenge",
    "motivation_level",
    "device_used_for_study"
]

for c in categorical_columns:
    print(f"Distinct values in {c}:")
    df.select(c).distinct().show(truncate=False)

Distinct values in education_level:
+---------------+
|education_level|
+---------------+
|Undergraduate  |
|Postgraduate   |
|Graduate       |
+---------------+

Distinct values in preferred_learning_method:
+-------------------------+
|preferred_learning_method|
+-------------------------+
|Video lectures           |
|Interactive discussion   |
|Recorded tutorials       |
|Reading notes            |
|Practice exercises       |
+-------------------------+

Distinct values in main_learning_challenge:
+---------------------------------+
|main_learning_challenge          |
+---------------------------------+
|Lack of concentration            |
|Internet connectivity issues     |
|Low motivation                   |
|Academic workload pressure       |
|Difficulty understanding concepts|
|Time management difficulty       |
+---------------------------------+

Distinct values in motivation_level:
+----------------+
|motivation_level|
+----------------+
|High            |
|Low             |
|

9. Get Value Counts for Each Categorical Column

In [42]:
for c in categorical_columns:
    print(f"Value counts for {c}:")
    df.groupBy(c) \
      .count() \
      .orderBy("count", ascending=False) \
      .show()

Value counts for education_level:
+---------------+-----+
|education_level|count|
+---------------+-----+
|       Graduate| 3368|
|   Postgraduate| 3320|
|  Undergraduate| 3312|
+---------------+-----+

Value counts for preferred_learning_method:
+-------------------------+-----+
|preferred_learning_method|count|
+-------------------------+-----+
|     Interactive discu...| 2040|
|           Video lectures| 2001|
|       Practice exercises| 1998|
|            Reading notes| 1997|
|       Recorded tutorials| 1964|
+-------------------------+-----+

Value counts for main_learning_challenge:
+-----------------------+-----+
|main_learning_challenge|count|
+-----------------------+-----+
|   Time management d...| 1738|
|   Lack of concentra...| 1716|
|   Internet connecti...| 1672|
|   Academic workload...| 1636|
|         Low motivation| 1620|
|   Difficulty unders...| 1618|
+-----------------------+-----+

Value counts for motivation_level:
+----------------+-----+
|motivation_level|count

10. Value Counts for Numerical Column (Study Hours Analysis)

In [43]:
df.groupBy("study_hours_per_day") \
  .count() \
  .orderBy("study_hours_per_day") \
  .show()

+-------------------+-----+
|study_hours_per_day|count|
+-------------------+-----+
|                  1| 1706|
|                  2| 1668|
|                  3| 1687|
|                  4| 1676|
|                  5| 1640|
|                  6| 1623|
+-------------------+-----+



11. Summary Statistics for Numerical Column

In [44]:
df.select("study_hours_per_day").describe().show()

+-------+-------------------+
|summary|study_hours_per_day|
+-------+-------------------+
|  count|              10000|
|   mean|             3.4745|
| stddev| 1.7054737213048505|
|    min|                  1|
|    max|                  6|
+-------+-------------------+



12. Preferred Learning Methods

In [45]:
df.groupBy("preferred_learning_method") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+-------------------------+-----+
|preferred_learning_method|count|
+-------------------------+-----+
|     Interactive discu...| 2040|
|           Video lectures| 2001|
|       Practice exercises| 1998|
|            Reading notes| 1997|
|       Recorded tutorials| 1964|
+-------------------------+-----+



13. Main Learning Challenges

In [46]:
df.groupBy("main_learning_challenge") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+-----------------------+-----+
|main_learning_challenge|count|
+-----------------------+-----+
|   Time management d...| 1738|
|   Lack of concentra...| 1716|
|   Internet connecti...| 1672|
|   Academic workload...| 1636|
|         Low motivation| 1620|
|   Difficulty unders...| 1618|
+-----------------------+-----+



14. Motivation Level Distribution

In [47]:
df.groupBy("motivation_level") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+----------------+-----+
|motivation_level|count|
+----------------+-----+
|          Medium| 3364|
|             Low| 3348|
|            High| 3288|
+----------------+-----+



15. Device Usage Analysis

In [48]:
df.groupBy("device_used_for_study") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+---------------------+-----+
|device_used_for_study|count|
+---------------------+-----+
|               Tablet| 3425|
|               Mobile| 3312|
|               Laptop| 3263|
+---------------------+-----+



16. Education Level vs Learning Method

In [49]:
df.groupBy("education_level", "preferred_learning_method") \
  .count() \
  .orderBy("education_level", "count", ascending=False) \
  .show()

+---------------+-------------------------+-----+
|education_level|preferred_learning_method|count|
+---------------+-------------------------+-----+
|  Undergraduate|            Reading notes|  666|
|  Undergraduate|           Video lectures|  664|
|  Undergraduate|       Practice exercises|  663|
|  Undergraduate|       Recorded tutorials|  662|
|  Undergraduate|     Interactive discu...|  657|
|   Postgraduate|     Interactive discu...|  692|
|   Postgraduate|           Video lectures|  680|
|   Postgraduate|            Reading notes|  675|
|   Postgraduate|       Recorded tutorials|  650|
|   Postgraduate|       Practice exercises|  623|
|       Graduate|       Practice exercises|  712|
|       Graduate|     Interactive discu...|  691|
|       Graduate|           Video lectures|  657|
|       Graduate|            Reading notes|  656|
|       Graduate|       Recorded tutorials|  652|
+---------------+-------------------------+-----+



17 Simple Sentiment Analysis Using VADER (PySpark)

In [50]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [51]:
vader = SentimentIntensityAnalyzer()


In [52]:
# an example use
vader.polarity_scores(
    "Online learning is flexible and convenient"
)

{'neg': 0.0, 'neu': 0.725, 'pos': 0.275, 'compound': 0.2263}

To run VADER sentiment analysis on a PySpark DataFrame, we need to wrap the VADER logic in a User Defined Function (UDF). Since VADER is a Python library, Spark needs this UDF to distribute the computation across our cluster.

In [53]:
from pyspark.sql.functions import udf, col, when
from pyspark.sql.types import FloatType

# Define the function to get the compound score
def get_vader_compound(text):
    if text is None:
        return 0.0
    return vader.polarity_scores(text)['compound']

# Register as a PySpark UDF
vader_udf = udf(get_vader_compound, FloatType())

Apply Transformation

In [54]:
# 1. Create the compound score column
df_with_scores = df.withColumn("compound_score", vader_udf(col("online_learning_opinion")))

# 2. Create the sentiment label column based on your rules
df = df_with_scores.withColumn(
    "sentiment_label",
    when(col("compound_score") > 0.05, "positive")
    .when(col("compound_score") < -0.05, "negative")
    .otherwise("neutral")
)

# Show the results
df.select("online_learning_opinion", "compound_score", "sentiment_label").show()

+-----------------------+--------------+---------------+
|online_learning_opinion|compound_score|sentiment_label|
+-----------------------+--------------+---------------+
|   Online learning i...|        0.2263|       positive|
|   Online learning i...|        0.4767|       positive|
|   Online learning i...|        0.4767|       positive|
|   Online platforms ...|           0.0|        neutral|
|   Online learning i...|        0.2263|       positive|
|   Blended learning ...|        0.5256|       positive|
|   Online learning r...|        0.5106|       positive|
|   Blended learning ...|        0.5256|       positive|
|   Online learning i...|        0.4767|       positive|
|   Blended learning ...|        0.5256|       positive|
|   Online learning i...|        0.2263|       positive|
|   Blended learning ...|        0.5256|       positive|
|   Online learning i...|        0.2263|       positive|
|   Online platforms ...|           0.0|        neutral|
|   Online learning i...|      

18 Getting 10 most common words in opinion column
We could use NLTk or spaCy NLP libraries to remove stopwords and lemmatize and then get the frequency but we kept it simple by providing a small list of stopwords.

In [55]:
# PySpark UDF version — returns top-10 words (no external NLP libs)
import re
from collections import Counter
from pyspark.sql.functions import col, udf, concat_ws
from pyspark.sql.types import ArrayType, StringType

STOPWORDS = {
    "the","and","is","in","to","of","a","for","on","it","this","that","i","you","we","they",
    "he","she","are","was","were","be","been","has","have","had","with","as","at","from","by",
    "an","or","not","but","if","then","so","my","your","their","our","me","us","them","which",
    "what","who","when","where","how","why","can","could","should","would","will","just","about",
    "like","also","than","because","more","most","some","any","each","other","only","into","out",
    "up","down","over","under","per","such","may","these","those"
}

def top_n_words_py(text, n=10):
    if text is None:
        return []
    words = re.findall(r"\b[a-zA-Z]{2,}\b", str(text).lower())
    filtered = [w for w in words if w not in STOPWORDS]
    return [w for w,_ in Counter(filtered).most_common(n)]

udf_top10 = udf(lambda t: top_n_words_py(t, 10), ArrayType(StringType()))

df = df.withColumn("top10_words", udf_top10(col("online_learning_opinion")))
df = df.withColumn("top10_words_str", concat_ws(", ", col("top10_words")))

19. Converting the spark dataframe to a pandas df so it can easily convert to csv

In [56]:
pandas_df = df.toPandas()
pandas_df.to_csv("processed.csv", index=False)

20. Stop Spark Session

In [57]:
spark.stop()